In [44]:
from dotenv import load_dotenv
import os
import clickhouse_connect
import pandas as pd


load_dotenv()

client = clickhouse_connect.get_client(
    host=os.getenv('CLICKHOUSE_HOST'),
    port=int(os.getenv('CLICKHOUSE_PORT', 8123)), 
    username=os.getenv('CLICKHOUSE_USERNAME'),
    password=os.getenv('CLICKHOUSE_PASSWORD'),
    database=os.getenv('CLICKHOUSE_DATABASE')
)

In [45]:
supplier_ranking_query = """SELECT
    v.vendor_name,
    SUM(CAST(ifNull(bni.quantity_in, 0) AS Float64)) AS total_quantity_received,
    ROUND(
        AVG(
            dateDiff(
                'day',
                parseDateTimeBestEffortOrNull(b.shipped_date),
                parseDateTimeBestEffortOrNull(b.receipt_date)
            )
        ), 2
    ) AS avg_days_for_shipment_to_arrive,
--    ROUND(
--        AVG(
--            dateDiff(
--                'day',
--                parseDateTimeBestEffortOrNull(b.shipped_date),
--                parseDateTimeBestEffortOrNull(b.bill_date)
--            )
--        ), 2
--    ) AS no_of_days_for_item_to_be_shipped,
    -- Net monetary transaction
    ROUND(
        SUM(
            CAST(
                replaceAll(replaceAll(b.total_bcy, 'USD ', ''), ',', '') AS Float64
            )
            -
            CAST(
                replaceAll(replaceAll(ifNull(b.discount_amount_bcy, '0'), 'USD ', ''), ',', '') AS Float64
            )
        ), 2
    ) AS net_monetary_transaction,
    -- On-time delivery rate
    ROUND(
        (countIf(
            parseDateTimeBestEffortOrNull(b.receipt_date) <= parseDateTimeBestEffortOrNull(b.eta)
        ) * 100.0)
        / nullIf(countIf(b.receipt_date != '' AND b.eta != ''), 0),
        2
    ) AS on_time_delivery_rate,
    -- Distinct products
    COUNT(DISTINCT bi.product_id) AS distinct_items_supplied,
COUNT(DISTINCT b.bill_id) AS total_shipments,
ROUND(
        COUNT(DISTINCT b.bill_id) 
        / nullIf(dateDiff('year', MIN(b.created_time), today()), 0),
        2
    ) AS avg_shipments_per_year
FROM zoho_books_analytics.batch_number_in bni
INNER JOIN zoho_books_analytics.bills b on bni.bill_id = b.bill_id 
INNER JOIN zoho_books_analytics.bill_item bi ON b.bill_id  = bi.bill_id
INNER JOIN zoho_books_analytics.purchase_orders p ON b.purchase_order   = p.purchase_order_number  
INNER JOIN zoho_books_analytics.items i ON bi.product_id   = i.item_id 
INNER JOIN zoho_books_analytics.sales_orders so ON p.reference_number = so.sales_order 
INNER JOIN zoho_books_analytics.customers c ON c.customer_id   = so.customer_id
INNER JOIN zoho_books_analytics.customer_item_mapping ci ON i.sku   = ci.az_sku 
INNER JOIN zoho_books_analytics.vendors v on v.vendor_id = b.vendor_id
WHERE	 c.customer_name   like 'Walmart%'
    AND b.receipt_date != '' 
    AND b.eta != ''
GROUP BY 
    v.vendor_name
ORDER BY 
    total_quantity_received DESC"""

In [46]:
result = client.query(query=supplier_ranking_query)

supplier_dataset_df = pd.DataFrame(result.result_rows, columns=[col for col in result.column_names])

In [47]:
supplier_dataset_df.fillna(100, inplace=True)

In [48]:
weights = {
    "total_quantity_received": 0.13,                # Total Quantity Received
    "avg_days_for_shipment_to_arrive": 0.13,        # Avg Shipment Transit Time
    "no_of_days_for_item_to_be_shipped": 0.08,      # Avg Shipment-to-Billing Duration
    "net_monetary_transaction": 0.13,               # Net Transaction Value
    "on_time_delivery_rate": 0.22,                  # On-Time Delivery Rate
    "distinct_items_supplied": 0.09,                # Distinct Items Supplied
    "total_shipments": 0.13,                        # Total Shipments
    "avg_shipments_per_year": 0.09                  # Avg Shipments per Year (replacing relationship duration)
}


In [49]:
supplier_dataset_df

,vendor_name,total_quantity_received,avg_days_for_shipment_to_arrive,net_monetary_transaction,on_time_delivery_rate,distinct_items_supplied,total_shipments,avg_shipments_per_year
0,Sandhya Aqua Exports Pvt Ltd,12311738.0,55.02,42831110.60,0.94,26,347,173.5
1,Aquatica Frozen Foods Global Pvt Ltd,4702140.0,61.84,21547392.72,0.00,16,96,48.0
2,Suryamitra Exim LTD,3145650.0,60.83,12962422.56,0.00,13,57,57.0
3,"Zalo Fresh, Inc.",2714728.0,46.05,9561150.80,0.00,6,62,31.0
4,SANDHYA MARINES LTD,2234124.0,61.90,9275962.36,0.00,13,55,27.5
5,Kalyan Aqua & Marine Exports India Pvt Ltd,1419794.0,77.36,4406209.20,0.00,9,32,16.0
6,PT KHOM FOODS,1194750.0,58.57,4837793.60,0.00,2,20,20.0
7,"NTSF Company, Inc",1109135.0,50.23,4296975.00,0.00,3,28,28.0
8,Geo Seafoods,1032000.0,64.07,3596765.60,6.67,4,25,25.0
9,PT FIRST MARINE SEAFOODS,840962.0,57.28,3558811.00,0.00,7,20,20.0


In [50]:
!pip install -q pymcdm

In [51]:
from pymcdm.methods import TOPSIS

# X = alternatives x criteria, weights = list, impacts = list
topsis = TOPSIS()
weights = [0.15, 0.15, 0.15, 0.22, 0.1, 0.13, 0.1]
# ↑ Matches: 
# total_quantity_received, avg_days_for_shipment_to_arrive, 
# net_monetary_transaction, on_time_delivery_rate, 
# distinct_items_supplied, total_shipments, avg_shipments_per_year

types = [1, -1, 1, 1, 1, 1, 1]
# 1  → higher is better
# -1 → lower is better (delay)

X = supplier_dataset_df.iloc[:, 1:].values

rank = topsis(X, weights=weights, types=types)


In [52]:
import numpy as np

In [53]:
supplier_dataset_df["score"] = np.round(rank * 100,2)

In [54]:
supplier_dataset_df.sort_values(by='score', ascending=False).to_csv("ranked_suppliers.csv", index=False)